# TrueID Live Commerce Copilot - Captioning Demo

This notebook runs the first MVP slice: timestamped captions for a live-commerce stream. It clones the GitHub repo when run from a blank Colab runtime. The default path uses a cached transcript so the demo completes without API keys, GPU, ffmpeg, or speech-model downloads.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Siratish/Live-Commerce-Copilot.git"
REPO_DIR_NAME = "Live-Commerce-Copilot"

def looks_like_project(path: Path) -> bool:
    return (path / "config" / "demo.yaml").exists() and (path / "src").exists()

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if looks_like_project(candidate):
        repo_root = candidate
        print(f"Using existing repo checkout: {repo_root}")
        break

if repo_root is None:
    clone_parent = Path("/content") if Path("/content").exists() else Path.cwd()
    repo_root = clone_parent / REPO_DIR_NAME
    if repo_root.exists() and not looks_like_project(repo_root):
        raise RuntimeError(f"{repo_root} exists but does not look like the target project.")
    if not repo_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo_root)])
    else:
        print(f"Using existing clone: {repo_root}")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Active repo root: {repo_root}")

requirements = repo_root / "requirements.txt"
if requirements.exists():
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
    except subprocess.CalledProcessError as exc:
        print(f"Dependency install failed, continuing with stdlib fallback: {exc}")
else:
    print("requirements.txt not found; continuing with the current runtime")

In [ ]:
from pathlib import Path
import json

from src.pipeline.run_captioning_demo import run_from_config

summary = run_from_config(Path("config/demo.yaml"))
summary

In [ ]:
caption_path = Path(summary["outputs"]["json"])
captions = json.loads(caption_path.read_text(encoding="utf-8"))

rows = []
for segment in captions["segments"]:
    rows.append({
        "start": segment["start"],
        "end": segment["end"],
        "seconds": round(segment["end"] - segment["start"], 2),
        "text": segment["text"],
        "source": segment["source"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    rows

In [ ]:
duration = captions.get("duration_seconds") or captions["segments"][-1]["end"]
bars = []
for index, segment in enumerate(captions["segments"], start=1):
    left = 100 * segment["start"] / duration
    width = 100 * (segment["end"] - segment["start"]) / duration
    bars.append(
        f'<div style="position:relative;height:26px;margin:4px 0;background:#f2f4f7;border-radius:4px;">'
        f'<div title="Segment {index}: {segment["text"]}" style="position:absolute;left:{left:.2f}%;width:{width:.2f}%;height:100%;background:#e51b23;border-radius:4px;"></div>'
        f'<span style="position:absolute;left:8px;top:4px;font:12px Arial;color:#111;">{index}</span>'
        f'</div>'
    )
html = '<h3>Caption Coverage Timeline</h3>' + ''.join(bars)
try:
    from IPython.display import HTML, display
    display(HTML(html))
except Exception:
    print(html)

In [ ]:
metrics_path = Path(summary["outputs"]["metrics"])
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
metrics

## Production Note

This first slice uses cached/file input for reliability. In production, the same `CaptioningEngine` interface can receive overlapping four-second audio chunks from an RTMP, HLS, or WebRTC adapter, publish WebVTT segments to the viewer overlay, and persist transcript events for moderator and post-show analytics.